# CelebAMask-HQ GPU Compare BG / HAIR / FACE 5k Prompt-Aligned Checkpoints

Use this notebook with a GPU runtime after the prompt-aligned BG/HAIR/FACE 5k micro-ablation run completes.
It compares the base model and prompt-aligned checkpoints with the same explicit layer prompt.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/CelebMaskHQ_Colab')
REPO_DRIVE = DRIVE_ROOT / 'repo'
PROCESSED_DRIVE_ROOT = DRIVE_ROOT / 'processed'
RUNS_DRIVE_ROOT = DRIVE_ROOT / 'runs'

PROJECT_LOCAL = Path('/content/project')
PACKAGE_LOCAL_ROOT = Path('/content/test_eval_package')

EVAL_OUTPUT_NAME = 'processed_celebmaskhq_bghairface_5k_test_eval_package'
EVAL_OUTPUT_DRIVE = PROCESSED_DRIVE_ROOT / EVAL_OUTPUT_NAME
PACKAGE_MANIFEST_DRIVE = EVAL_OUTPUT_DRIVE / 'package_manifest.json'

MICRO_RUN_NAME = 'qwen_layered_lora_bghairface_5k_prompt_aligned_micro_ablation'
MICRO_RUN_DRIVE = RUNS_DRIVE_ROOT / MICRO_RUN_NAME
MICRO_RUN_LOCAL = Path(f'/content/{MICRO_RUN_NAME}')

COMPARISON_RUN_NAME = 'qwen_layered_lora_bghairface_5k_prompt_aligned_micro_ablation_compare'
COMPARISON_DRIVE = RUNS_DRIVE_ROOT / COMPARISON_RUN_NAME
COMPARISON_LOCAL = Path(f'/content/{COMPARISON_RUN_NAME}')

MODEL_VARIANTS = [
    {
        'name': 'base_model',
        'type': 'base',
    },
    {
        'name': 'checkpoint_25',
        'type': 'lora',
        'run_dir': MICRO_RUN_LOCAL,
        'checkpoint_name': 'checkpoint-25',
    },
    {
        'name': 'checkpoint_50',
        'type': 'lora',
        'run_dir': MICRO_RUN_LOCAL,
        'checkpoint_name': 'checkpoint-50',
    },
    {
        'name': 'checkpoint_75',
        'type': 'lora',
        'run_dir': MICRO_RUN_LOCAL,
        'checkpoint_name': 'checkpoint-75',
    },
]
MAX_COMPARE_SAMPLES = 3
MIXED_PRECISION = 'bf16'
RESOLUTION = 640
MAX_LAYERS = 3
INFERENCE_STEPS = 50
TRUE_CFG_SCALE = 4.0
SEED = 1337
FORCE_REBUILD_COMPARISON = True
PROMPT = 'decompose this portrait into exactly three editable RGBA layers: background, hair, and face'

print('Repo on Drive:', REPO_DRIVE)
print('Eval package on Drive:', EVAL_OUTPUT_DRIVE)
print('Micro-ablation alpha-weighted run on Drive:', MICRO_RUN_DRIVE)
print('Comparison output on Drive:', COMPARISON_DRIVE)
print('Prompt:', PROMPT)


In [ ]:
!nvidia-smi


In [ ]:
!pip uninstall -y torchao
!pip install -U accelerate peft bitsandbytes sentencepiece protobuf
!pip install -U "transformers>=4.51.3"
!pip install git+https://github.com/huggingface/diffusers.git


In [ ]:
import json
import shutil
import subprocess

assert REPO_DRIVE.exists(), f'Missing repo folder on Drive: {REPO_DRIVE}'
assert PACKAGE_MANIFEST_DRIVE.exists(), f'Missing package manifest on Drive: {PACKAGE_MANIFEST_DRIVE}'
assert MICRO_RUN_DRIVE.exists(), f'Missing mirrored micro-ablation alpha-weighted run on Drive: {MICRO_RUN_DRIVE}'

package_manifest = json.loads(PACKAGE_MANIFEST_DRIVE.read_text(encoding='utf-8'))
assert package_manifest['target_split'] == 'test', package_manifest['target_split']

PACKAGE_NAME = package_manifest['package_name']
METADATA_TAR_DRIVE = EVAL_OUTPUT_DRIVE / package_manifest['metadata_tar']
DATA_TAR_DRIVE = EVAL_OUTPUT_DRIVE / package_manifest['data_tar']
EVAL_LOCAL = Path('/content') / PACKAGE_NAME
METADATA_TAR_LOCAL = PACKAGE_LOCAL_ROOT / package_manifest['metadata_tar']
DATA_TAR_LOCAL = PACKAGE_LOCAL_ROOT / package_manifest['data_tar']

if PROJECT_LOCAL.exists():
    shutil.rmtree(PROJECT_LOCAL)
if PACKAGE_LOCAL_ROOT.exists():
    shutil.rmtree(PACKAGE_LOCAL_ROOT)
if EVAL_LOCAL.exists():
    shutil.rmtree(EVAL_LOCAL)
if MICRO_RUN_LOCAL.exists():
    shutil.rmtree(MICRO_RUN_LOCAL)
if COMPARISON_LOCAL.exists():
    shutil.rmtree(COMPARISON_LOCAL)
if FORCE_REBUILD_COMPARISON and COMPARISON_DRIVE.exists():
    shutil.rmtree(COMPARISON_DRIVE)

PACKAGE_LOCAL_ROOT.mkdir(parents=True, exist_ok=True)
shutil.copytree(REPO_DRIVE, PROJECT_LOCAL)
shutil.copytree(MICRO_RUN_DRIVE, MICRO_RUN_LOCAL)
shutil.copy2(METADATA_TAR_DRIVE, METADATA_TAR_LOCAL)
shutil.copy2(DATA_TAR_DRIVE, DATA_TAR_LOCAL)

for command in [
    ['tar', '-xf', str(METADATA_TAR_LOCAL), '-C', '/content'],
    ['tar', '-xf', str(DATA_TAR_LOCAL), '-C', '/content'],
]:
    print('Running extract command:', ' '.join(command))
    subprocess.run(command, check=True)

assert EVAL_LOCAL.exists(), f'Eval package was not extracted: {EVAL_LOCAL}'
assert (EVAL_LOCAL / 'metadata' / 'layered_samples.jsonl').exists(), (
    f'Missing layered metadata after extract: {EVAL_LOCAL / "metadata" / "layered_samples.jsonl"}'
)

print('Copied repo to', PROJECT_LOCAL)
print('Copied micro-ablation alpha-weighted run to', MICRO_RUN_LOCAL)
print('Extracted eval package to', EVAL_LOCAL)


In [ ]:
import json

layered_manifest_path = EVAL_LOCAL / 'metadata' / 'layered_samples.jsonl'
all_rows = [json.loads(line) for line in layered_manifest_path.read_text(encoding='utf-8').splitlines() if line.strip()]
row_splits = sorted(set(row['split'] for row in all_rows))
assert row_splits == ['test'], row_splits
assert all(row['layer_names'] == ['BG', 'HAIR', 'FACE'] for row in all_rows), all_rows[0]
assert all(row['layer_count'] == 3 for row in all_rows), all_rows[0]

if len(all_rows) <= MAX_COMPARE_SAMPLES:
    rows = all_rows
else:
    if MAX_COMPARE_SAMPLES == 1:
        selected_indices = [len(all_rows) // 2]
    else:
        last_index = len(all_rows) - 1
        selected_indices = [round(i * last_index / (MAX_COMPARE_SAMPLES - 1)) for i in range(MAX_COMPARE_SAMPLES)]
    rows = [all_rows[index] for index in sorted(dict.fromkeys(selected_indices))]
    while len(rows) < MAX_COMPARE_SAMPLES:
        rows.append(all_rows[len(rows)])

for variant in MODEL_VARIANTS:
    if variant['type'] == 'base':
        continue
    checkpoint_dir = variant['run_dir'] / variant['checkpoint_name']
    assert checkpoint_dir.exists(), f'Missing checkpoint directory: {checkpoint_dir}'
    assert (checkpoint_dir / 'pytorch_lora_weights.safetensors').exists(), checkpoint_dir

print('Eval manifest split check passed:', row_splits)
print('All packaged test sample ids:', [int(row['sample_id']) for row in all_rows])
print('Selected compare sample ids:', [int(row['sample_id']) for row in rows])
print('Model variants:', [variant['name'] for variant in MODEL_VARIANTS])


In [ ]:
import json
import shutil
import torch
from diffusers import QwenImageLayeredPipeline
from PIL import Image

assert torch.cuda.is_available(), 'This checkpoint comparison requires a GPU runtime.'

comparison_summary = []
if COMPARISON_LOCAL.exists():
    shutil.rmtree(COMPARISON_LOCAL)
COMPARISON_LOCAL.mkdir(parents=True, exist_ok=True)

for variant in MODEL_VARIANTS:
    variant_name = variant['name']
    checkpoint_dir = variant.get('run_dir', MICRO_RUN_LOCAL) / variant['checkpoint_name'] if variant['type'] == 'lora' else None
    source_label = checkpoint_dir if checkpoint_dir is not None else 'Qwen/Qwen-Image-Layered base model'
    print(f"Running inference for {variant_name} from {source_label}")
    torch.cuda.empty_cache()
    pipeline = QwenImageLayeredPipeline.from_pretrained('Qwen/Qwen-Image-Layered')
    pipeline = pipeline.to('cuda', torch.bfloat16)
    pipeline.set_progress_bar_config(disable=None)
    if checkpoint_dir is not None:
        pipeline.load_lora_weights(str(checkpoint_dir), weight_name='pytorch_lora_weights.safetensors')

    for row in rows:
        sample_id = f"{int(row['sample_id']):05d}"
        composite_path = EVAL_LOCAL / row['composite_path']
        input_image = Image.open(composite_path).convert('RGBA')
        requested_layers = int(row.get('layer_count') or len(row.get('layer_paths', [])) or 1)
        requested_layers = max(1, min(requested_layers, MAX_LAYERS))

        sample_output_dir = COMPARISON_LOCAL / variant_name / sample_id
        sample_output_dir.mkdir(parents=True, exist_ok=True)
        input_save_path = sample_output_dir / 'input.png'
        input_image.save(input_save_path)
        (sample_output_dir / 'selected_sample.json').write_text(
            json.dumps(row, indent=2, sort_keys=True),
            encoding='utf-8',
        )
        (sample_output_dir / 'variant.json').write_text(
            json.dumps({
                'name': variant_name,
                'type': variant['type'],
                'checkpoint_name': variant.get('checkpoint_name'),
            'prompt': PROMPT,
                'prompt': PROMPT,
            }, indent=2, sort_keys=True),
            encoding='utf-8',
        )

        with torch.inference_mode():
            output = pipeline(
                image=input_image,
                prompt=PROMPT,
                generator=torch.Generator(device='cuda').manual_seed(SEED),
                true_cfg_scale=TRUE_CFG_SCALE,
                negative_prompt=' ',
                num_inference_steps=INFERENCE_STEPS,
                num_images_per_prompt=1,
                layers=requested_layers,
                resolution=RESOLUTION,
                cfg_normalize=True,
                use_en_prompt=True,
            )

        predicted_layers = output.images[0]
        assert predicted_layers, f'Pipeline returned no layer images for {variant_name} sample {sample_id}'
        saved_layer_paths = []
        for layer_index, image in enumerate(predicted_layers):
            output_path = sample_output_dir / f'layer_{layer_index:02d}.png'
            image.save(output_path)
            saved_layer_paths.append(str(output_path))

        comparison_summary.append({
            'variant': variant_name,
            'type': variant['type'],
            'checkpoint_name': variant.get('checkpoint_name'),
            'prompt': PROMPT,
            'sample_id': sample_id,
            'requested_layers': requested_layers,
            'predicted_layers': len(saved_layer_paths),
            'output_dir': str(sample_output_dir),
        })
        print(f"  sample {sample_id}: requested={requested_layers} predicted={len(saved_layer_paths)}")

    del pipeline
    torch.cuda.empty_cache()

(COMPARISON_LOCAL / 'comparison_summary.json').write_text(
    json.dumps(comparison_summary, indent=2, sort_keys=True),
    encoding='utf-8',
)

RUNS_DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
if COMPARISON_DRIVE.exists():
    shutil.rmtree(COMPARISON_DRIVE)
shutil.copytree(COMPARISON_LOCAL, COMPARISON_DRIVE)

print('Checkpoint comparison finished.')
print('Saved local outputs to:', COMPARISON_LOCAL)
print('Saved Drive outputs to:', COMPARISON_DRIVE)


In [ ]:
for row in comparison_summary:
    print(json.dumps(row, indent=2, sort_keys=True))


## Result

Review the saved comparison directories on Drive to judge whether the explicit
`background, hair, and face` prompt improves slot binding over the generic-prompt run.
